In [59]:
import numpy as np
import pandas as pd


class UserBasedCF:

    def __init__(self, user_item_matrix, top_k=20, min_common=10):

        self.user_item = user_item_matrix
        self.top_k = top_k
        self.min_common = min_common
        self.matrix=self.user_item.to_numpy()

        # Cache
        self.neighbor_cache = {}
        self.user_means = self.user_item.mean(axis=1)
        
        self.user_to_idx = {
            user: i
            for i, user in enumerate(self.user_item.index)
        }

        self.movie_to_idx = {
            movie: j
            for j, movie in enumerate(self.user_item.columns)
        }
        
        
        self.user_mean_array = self.user_means.to_numpy()
        
    def fit(self):
        print("Training UserCF...")
        self.user_means = self.user_item.mean(axis=1)
        
        # similarity matrix
        self.similarity_matrix = pd.DataFrame(
            index=self.user_item.index,
            columns=self.user_item.index,
            dtype=float
        )
        
        # nested loop to calculate similarity between all pairs of users
        for i, user1 in enumerate(self.user_item.index):
            for j, user2 in enumerate(self.user_item.index):

                if j < i:
                    continue

                similarity, common = self.pearson_similarity(
                    self.user_item.loc[user1],
                    self.user_item.loc[user2]
                )

                self.similarity_matrix.loc[user1, user2] = similarity
                self.similarity_matrix.loc[user2, user1] = similarity
                
                
                
    # ==========================================================
    # Pearson Correlation
    # ==========================================================

    def pearson_similarity(self, user1, user2):

        common = user1.notna() & user2.notna()

        common_count = common.sum()

        if common_count < self.min_common:
            return 0.0, common_count

        u1 = user1[common]
        u2 = user2[common]

        u1=u1-u1.mean()
        u2=u2-u2.mean()


        # we cannot use the user means here because we are only considering the common ratings between the two users. The user means are calculated over all ratings, not just the common ones. Therefore, we need to subtract the mean of the common ratings for each user instead of the overall user mean.
        
        # u1 = u1 - self.user_means[user1.name]
        # u2 = u2 - self.user_means[user2.name]

        norm1 = np.linalg.norm(u1)
        norm2 = np.linalg.norm(u2)

        if norm1 == 0 or norm2 == 0:
            return 0.0, common_count

        similarity = np.dot(u1, u2) / (norm1 * norm2)

        return similarity, common_count
    
    
    
    # ==========================================================
    # Find Candidate Movies
    # ==========================================================
    def get_candidate_movies(
        self,
        target_user,
        top_k=None
    ):

        neighbors = self.get_neighbors(
            target_user,
            top_k
        )

        watched_movies = set(
            self.user_item.loc[target_user]
            .dropna()
            .index
        )

        candidate_movies = set()

        for neighbor_id, similarity, common in neighbors:

            neighbor_movies = (
                self.user_item
                .loc[neighbor_id]
                .dropna()
                .index
            )

            candidate_movies.update(neighbor_movies)

        candidate_movies -= watched_movies

        return list(candidate_movies)
    
    
    # ==========================================================
    # Find Similar Users
    # ==========================================================

    def get_neighbors(self, target_user, top_k=None):

        if top_k is None:
            top_k = self.top_k

        if target_user in self.neighbor_cache:
            return self.neighbor_cache[target_user]

        target = self.user_item.loc[target_user]
        neighbors = []

        for other_user in self.user_item.index:

            if other_user == target_user:
                continue

            similarity, common = self.pearson_similarity(
                target,
                self.user_item.loc[other_user]
            )

            if similarity <= 0:
                continue

            neighbors.append(
                (
                    other_user,
                    similarity,
                    common
                )
            )

        neighbors.sort(
            key=lambda x:x[1],
            reverse=True
        )

        neighbors = neighbors[:top_k]

        self.neighbor_cache[target_user] = neighbors

        return neighbors

    # ==========================================================
    # Predict Rating
    # ==========================================================

    def predict_rating(
        self,
        target_user,
        target_movie,
        top_k=None
    ):

        if top_k is None:
            top_k = self.top_k

        neighbors = self.get_neighbors(
            target_user,
            top_k
        )

        # target_mean = self.user_means[target_user]
        target_idx = self.user_to_idx[target_user]
        target_mean = self.user_mean_array[target_idx]
        
        numerator = 0.0
        denominator = 0.0

        for neighbor_id, similarity, common in neighbors:

            # this is culprit it takes very long time to access the dataframe for each neighbor and movie. Instead, we can use the precomputed matrix to get the rating directly.
            # rating = self.user_item.loc[
            #     neighbor_id,
            #     target_movie
            # ]


            u = self.user_to_idx[neighbor_id]
            m = self.movie_to_idx[target_movie]

            rating = self.matrix[u, m]
            if pd.isna(rating):
                continue

            # neighbor_mean = self.user_means[neighbor_id]

            neighbor_mean = self.user_mean_array[u]
            deviation = rating - neighbor_mean

            numerator += similarity * deviation
            denominator += similarity

        if denominator == 0:
            return None

        prediction = target_mean + (numerator / denominator)

        prediction = max(1, min(5, prediction))

        return prediction

    # ==========================================================
    # Recommend Movies
    # ==========================================================

    def recommend(
        self,
        target_user,
        top_n=10,
        top_k=None
    ):

        if top_k is None:
            top_k = self.top_k

        recommendations = []

        candidate_movies = self.get_candidate_movies(
            target_user,
            top_k
        )

        for movie in candidate_movies:

            prediction = self.predict_rating(
                target_user,
                movie,
                top_k
            )

            if prediction is not None:

                recommendations.append({
                "movie_id": movie,
                "predicted_rating": prediction
            })

        recommendations.sort(
        key=lambda x: x["predicted_rating"],
        reverse=True
)

        return recommendations[:top_n]

    # ==========================================================
    # Evaluate
    # ==========================================================

    def evaluate(self, test_df):

        predictions = []
        squared_errors = []
        absolute_errors = []

        for _, row in test_df.iterrows():

            user = row["user_id"]
            movie = row["movie_id"]
            actual = row["rating"]

            if user not in self.user_item.index:
                continue

            if movie not in self.user_item.columns:
                continue

            predicted = self.predict_rating(
                user,
                movie
            )

            if predicted is None:
                continue

            predictions.append(
                {
                    "user_id": user,
                    "movie_id": movie,
                    "actual": actual,
                    "predicted": predicted,
                    "error": abs(actual - predicted)
                }
            )

            squared_errors.append(
                (actual - predicted) ** 2
            )

            absolute_errors.append(
                abs(actual - predicted)
            )

        prediction_df = pd.DataFrame(predictions)

        rmse = np.sqrt(np.mean(squared_errors))
        mae = np.mean(absolute_errors)

        return prediction_df, rmse, mae

In [60]:
train = pd.read_csv(
    "data/u1.base",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

test = pd.read_csv(
    "data/u1.test",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

In [61]:
train_matrix = train.pivot(
    index="user_id",
    columns="movie_id",
    values="rating"
)

In [62]:
cf = UserBasedCF(
    train_matrix,
    top_k=20,
    min_common=10
)

In [63]:
cf.get_neighbors(1)

[(69, np.float64(0.8448192338623107), np.int64(14)),
 (329, np.float64(0.8416650799680397), np.int64(11)),
 (320, np.float64(0.7957040966440955), np.int64(16)),
 (115, np.float64(0.7863336509949342), np.int64(12)),
 (21, np.float64(0.7690614556304467), np.int64(12)),
 (526, np.float64(0.6826021514709876), np.int64(11)),
 (22, np.float64(0.6780500082363685), np.int64(17)),
 (413, np.float64(0.6714450198148366), np.int64(11)),
 (197, np.float64(0.6648959685418467), np.int64(10)),
 (291, np.float64(0.6642456559897918), np.int64(32)),
 (517, np.float64(0.6442227924869779), np.int64(10)),
 (294, np.float64(0.6421554612114082), np.int64(14)),
 (246, np.float64(0.6259701640216627), np.int64(24)),
 (623, np.float64(0.6239181323861149), np.int64(13)),
 (540, np.float64(0.6161469221192538), np.int64(19)),
 (826, np.float64(0.6077192928383598), np.int64(25)),
 (82, np.float64(0.6015633408367517), np.int64(21)),
 (275, np.float64(0.6000192009216492), np.int64(11)),
 (890, np.float64(0.595189895870

In [64]:
cf.predict_rating(
    target_user=1,
    target_movie=300
)

np.float64(3.801693681033199)

In [65]:
import time
start = time.time()
recommendations = cf.recommend(
    target_user=1,
    top_n=10
)
print("Time taken for recommendations: ", time.time() - start)
recommendations

Time taken for recommendations:  0.012420177459716797


[{'movie_id': 12, 'predicted_rating': 5},
 {'movie_id': 201, 'predicted_rating': 5},
 {'movie_id': 303, 'predicted_rating': 5},
 {'movie_id': 315, 'predicted_rating': 5},
 {'movie_id': 330, 'predicted_rating': 5},
 {'movie_id': 347, 'predicted_rating': 5},
 {'movie_id': 433, 'predicted_rating': 5},
 {'movie_id': 483, 'predicted_rating': 5},
 {'movie_id': 509, 'predicted_rating': 5},
 {'movie_id': 616, 'predicted_rating': 5}]

In [ ]:

start = time.time()
prediction_df, rmse, mae = cf.evaluate(test)
print("Time taken for evaluation: ", time.time() - start)

print("MAE :", mae)
print("RMSE:", rmse)

prediction_df.head()

In [ ]:
%load_ext line_profiler

In [ ]:
%lprun -f cf.predict_rating cf.evaluate(test.head(500))

In [ ]:
import cProfile

cProfile.run("cf.evaluate(test.head(500))")